# Fetching LRRK2 raw genotypes

## Loading Python libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import re
from io import StringIO


## Set variables

In [ ]:
REL11_PATH = pathlib.Path(pathlib.Path.home(), "workspace", "gp2_tier2_eu_release11")
CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, "clinical_data", "master_key_release11_final_vwb.csv")
RELATED_DATA_PATH = pathlib.Path(REL11_PATH, "meta_data", "related_samples")
IMPUTED_GENO_PATH = pathlib.Path(REL11_PATH, "imputed_genotypes")
WGS_PATH = pathlib.Path(REL11_PATH, "wgs", "deepvariant_joint_calling", "plink")
EXOME_PATH = pathlib.Path(REL11_PATH, "clinical_exomes", "plink")
RAW_GENO_PATH_FLIPPED = pathlib.Path(REL11_PATH, "raw_genotypes_flipped")

WORK_DIR = pathlib.Path(pathlib.Path.home(), "workspace", "ws_files", "results")


In [ ]:
ancestries = [
    "AAC", 
    "AFR", 
    "AJ", 
    "AMR", 
    "CAH", 
    "CAS", 
    "EAS", 
    "FIN", 
    "MDE", 
    "SAS",
    "EUR",
]
ancestry_mappings = {
    "AAC": "Admixed African (AAC)", 
    "AFR": "African (AFR)", 
    "AJ": "Ashkenazi Jewish (AJ)", 
    "AMR": "Admixed American (AMR)", 
    "CAH": "Complex Admixture (CAH)", 
    "CAS": "Central Asian (CAS)", 
    "EAS": "East Asian (EAS)", 
    "FIN": "Finnish (FIN)", 
    "MDE": "Middle Eastern (MDE)", 
    "SAS": "South Asian (SAS)",
    "EUR": "European (EUR)",
}
modalities = [
    "raw",
    "imputed",
    "wgs",
]
chrnum = 12
from_bp = 40225618
to_bp = 40367760


## Install Packages

In [ ]:
%%bash

if test -e /home/jupyter/plink1.9; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter/
    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 
    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
    chmod u+x ./plink1.9
    rm ./plink*.zip
fi


In [ ]:
%%bash

if test -e /home/jupyter/plink2; then
    echo "Plink2 is already installed in /home/jupyter/"
else
    echo "Plink2 is not installed"
    cd /home/jupyter/
    wget http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
    unzip -o plink2_linux_x86_64_latest.zip
    chmod u+x ./plink2
    rm ./plink*.zip*
fi


In [ ]:
%%bash

if test -e /home/jupyter/annovar; then
    echo "ANNOVAR is already installed in /home/jupyter/"
else
    echo "ANNOVAR is not installed, installing now..."
    cd /home/jupyter/
    wget http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz
    tar xvfz annovar.latest.tar.gz

    cd /home/jupyter/annovar
    perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb/
    perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
    perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/
    perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
    perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar avsnp151 humandb/

    chmod +x *.pl

    rm /home/jupyter/annovar*.tar.gz
fi


In [ ]:
if not pathlib.Path(f"{WORK_DIR}/hg38.fa.gz").exists():
    ! wget -P {WORK_DIR} "https://hgdownload.soe.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz" 


# Fetch LRRK2 Genotypes

## Load covariates

In [ ]:
df_cov = pd.read_csv(CLINICAL_DATA_PATH)[[
    "GP2ID",
    "age_of_onset",
    "age_at_diagnosis",
    "baseline_GP2_phenotype_for_qc",
    "family_history_for_qc",
    "baseline_GP2_phenotype",
    "age_at_death",
    "age_at_last_follow_up",
    "age_at_sample_collection",
]].rename(columns={"GP2ID":"IID", "baseline_GP2_phenotype_for_qc":"PHENO"})
df_cov.loc[df_cov["baseline_GP2_phenotype"] == "Affected_PD", "PHENO"] = "PD"
df_cov.loc[df_cov["baseline_GP2_phenotype"] == "Population Control", "PHENO"] = "Control"
df_cov["age_of_onset"] = df_cov["age_of_onset"].fillna(df_cov["age_at_diagnosis"])
df_cov["current_age"] = df_cov["age_at_death"].fillna(df_cov["age_at_last_follow_up"]).fillna(df_cov["age_at_sample_collection"])
df_cov = df_cov[(df_cov["age_of_onset"] >= 10) | (df_cov["age_of_onset"].isna())]
df_cov = df_cov[(df_cov["age_of_onset"] <= 100) | (df_cov["age_of_onset"].isna())]
df_cov.drop(columns=[
    "baseline_GP2_phenotype", 
    "age_at_diagnosis", 
    "age_at_death", 
    "age_at_last_follow_up", 
    "age_at_sample_collection",
], inplace=True)


## Keep only unrelated cases and controls

### Keep only cases and controls

In [ ]:
df_cov = df_cov[df_cov["PHENO"].isin(["PD", "Control"])]


### Remove related samples

In [ ]:
df_related = []
for ancestry in ancestries:
    df_related_ancestry = pd.read_csv(f'{RELATED_DATA_PATH}/{ancestry}_release11_vwb.related')
    df_related_ancestry = df_related_ancestry[
        (df_related_ancestry["IID1"].isin(df_cov["IID"].tolist())) & 
        (df_related_ancestry["IID2"].isin(df_cov["IID"].tolist()))
    ]
    df_related.append(df_related_ancestry)
df_related = pd.concat(df_related)
df_cov_unrelated = df_cov[~df_cov["IID"].isin(list(df_related["IID1"]))].copy()


### Save final list of participants to include 

In [ ]:
df_pheno = df_cov_unrelated[["IID", "PHENO"]].copy()
df_pheno.insert(0, "#FID", df_cov_unrelated["IID"].tolist())
df_pheno.loc[df_pheno["PHENO"] == "Control", "PHENO"] = 1
df_pheno.loc[df_pheno["PHENO"] == "PD", "PHENO"] = 2
df_pheno.to_csv(f"{WORK_DIR}/pheno.tsv", sep="\t", index=False)
df_pheno[["#FID", "IID"]].to_csv(f"{WORK_DIR}/sample_ids.tsv", sep="\t", index=False)


## Extract LRRK2, align to reference genome

### All other modalities

In [ ]:
for ancestry in ancestries:
    modality_prefixes = {
        "raw": f"{RAW_GENO_PATH_FLIPPED}/{ancestry}/{ancestry}_release11_flipped_vwb",
        "imputed": f"{IMPUTED_GENO_PATH}/{ancestry}/chr{chrnum}_{ancestry}_release11_vwb",
        "wgs": f"{WGS_PATH}/{ancestry}/chr{chrnum}_{ancestry}_release11",
    }
    for modality in modality_prefixes:
        if not pathlib.Path(f"{WORK_DIR}/results/{ancestry}/LRRK2_{modality}_{ancestry}.pgen").exists():
            pathlib.Path(WORK_DIR, "results", ancestry).mkdir(parents=True, exist_ok=True)

            cmd = f"/home/jupyter/plink2 "
            cmd += f"--pfile {modality_prefixes[modality]} "
            cmd += f"--fa {WORK_DIR}/hg38.fa.gz "
            cmd += f"--ref-from-fa force "
            cmd += f'--set-all-var-ids "chr@:#:\\$r:\\$a" '
            cmd += f"--new-id-max-allele-len 999 "
            cmd += f"--sort-vars "
            cmd += f"--chr {chrnum} "
            cmd += f"--from-bp {from_bp} "
            cmd += f"--to-bp {to_bp} "
            cmd += f"--keep {WORK_DIR}/sample_ids.tsv "
            cmd += f"--pheno {WORK_DIR}/pheno.tsv "
            cmd += f"--pheno-name PHENO "
            cmd += f"--no-categorical "
            if modality == "imputed":
                cmd += f'--extract-if-info "R2 >= 0.8" '
            cmd += f"--make-pgen "
            cmd += f"--out {WORK_DIR}/results/{ancestry}/LRRK2_{modality}_{ancestry} "
            
            !{cmd}
    
            cmd = f"""
            /home/jupyter/plink2 \
                --pfile {WORK_DIR}/results/{ancestry}/LRRK2_{modality}_{ancestry} \
                --mac 1 \
                --set-all-var-ids "chr@:#:\\$r:\\$a" \
                --new-id-max-allele-len 999 \
                --export A \
                --out {WORK_DIR}/results/{ancestry}/LRRK2_{modality}_{ancestry}
            """
            
            !{cmd}


## Read and reformat genotypes

In [ ]:
df_geno_dict = {}
for ancestry in ancestries:
    # Load in raw genotyping files, keeping only relevant parts of variant IDs.
    df_geno_dict[ancestry] = {}
    modality_raw_paths = {
        "raw":f"{WORK_DIR}/results/{ancestry}/LRRK2_raw_{ancestry}.raw",
        "imputed":f"{WORK_DIR}/results/{ancestry}/LRRK2_imputed_{ancestry}.raw",
        "wgs":f"{WORK_DIR}/results/{ancestry}/LRRK2_wgs_{ancestry}.raw",
    }
    for modality in modalities:
        df_geno = pd.read_csv(f"{WORK_DIR}/results/{ancestry}/LRRK2_{modality}_{ancestry}.raw", sep="\t")
        df_geno.columns = [col.split("_")[0] for col in df_geno.columns]
        df_geno.drop(columns=["FID","PAT","MAT","SEX"], inplace=True)
        df_geno_dict[ancestry][modality] = df_geno


## Process genotype data

In [ ]:
var_counts_dict = {}
for modality in modalities:
    var_counts_dict[modality] = []
    for ancestry in ancestries:
        df_geno = df_geno_dict[ancestry][modality]
        
        # Precompute totals across the whole dataset for phenotype
        n_controls = (df_geno["PHENOTYPE"] == 1).sum()
        n_cases = (df_geno["PHENOTYPE"] == 2).sum()
        
        # Initialize result dict
        var_counts = {
            "ID": [],
            "het_PD": [],
            "hom_PD": [],
            "total_PD": [n_cases] * (df_geno.shape[1] - 2),
            "het_HC": [],
            "hom_HC": [],
            "total_HC": [n_controls] * (df_geno.shape[1] - 2),
        }
        
        # Loop through SNPs and calculate counts
        for snp in [col for col in df_geno.columns.values if not col in ["IID", "PHENOTYPE"]]:
            var_counts["ID"].append(snp)
            var_counts["het_PD"].append(((df_geno["PHENOTYPE"] == 2) & (df_geno[snp] >= 0.5) & (df_geno[snp] <= 1.5)).sum())
            var_counts["hom_PD"].append(((df_geno["PHENOTYPE"] == 2) & (df_geno[snp] < 0.5)).sum())
            var_counts["het_HC"].append(((df_geno["PHENOTYPE"] == 1) & (df_geno[snp] >= 0.5) & (df_geno[snp] <= 1.5)).sum())
            var_counts["hom_HC"].append(((df_geno["PHENOTYPE"] == 1) & (df_geno[snp] < 0.5)).sum())

        # Convert results to DataFrame
        df_var_counts = pd.DataFrame(var_counts)
        df_var_counts["Ancestry"] = ancestry
        var_counts_dict[modality].append(df_var_counts)
    var_counts_dict[modality] = pd.concat(var_counts_dict[modality])
        

## Analyze exome data (no ancestry predictions, no controls)

In [ ]:
if not pathlib.Path(f"{WORK_DIR}/results/exomes/LRRK2.pgen").exists():
    pathlib.Path(WORK_DIR, "results", "exomes").mkdir(parents=True, exist_ok=True)
    
    cmd = f"""
    /home/jupyter/plink2 \
        --pfile {EXOME_PATH}/chr{chrnum} \
        --fa {WORK_DIR}/hg38.fa.gz \
        --ref-from-fa force \
        --set-all-var-ids "chr@:#:\\$r:\\$a" \
        --new-id-max-allele-len 999 \
        --sort-vars \
        --chr {chrnum} \
        --from-bp {from_bp} \
        --to-bp {to_bp} \
        --mac 1 \
        --keep {WORK_DIR}/sample_ids.tsv \
        --pheno {WORK_DIR}/pheno.tsv \
        --pheno-name PHENO \
        --no-categorical \
        --make-pgen \
        --out {WORK_DIR}/results/exomes/LRRK2
    """
    
    !{cmd}
    
    cmd = f"""
    /home/jupyter/plink2 \
        --pfile {WORK_DIR}/results/exomes/LRRK2 \
        --set-all-var-ids "chr@:#:\\$r:\\$a" \
        --new-id-max-allele-len 999 \
        --export A \
        --out {WORK_DIR}/results/exomes/LRRK2
    """
    
    !{cmd}


In [ ]:
df_geno_exome = pd.read_csv(f"{WORK_DIR}/results/exomes/LRRK2.raw", sep="\t")
df_geno_exome.columns = [col.split("_")[0] for col in df_geno_exome.columns]
df_geno_exome.drop(columns=["FID","PAT","MAT","SEX"], inplace=True)


# Process metadata for each variant

## For each variant, find list of carriers across each modality from each ancestry

In [ ]:
carrier_lists = {}
for ancestry in ancestries:
    for modality in modality_raw_paths:
        df_geno = df_geno_dict[ancestry][modality].drop(columns="PHENOTYPE")
        for variant in [c for c in df_geno.columns if c != "IID"]:
            new_participants = df_geno.loc[df_geno[variant] < 2, "IID"].tolist()
            if len(new_participants) > 0:
                if variant not in carrier_lists:
                    carrier_lists[variant] = new_participants
                else:
                    carrier_lists[variant] = list(set(carrier_lists[variant] + new_participants))

for variant in [c for c in df_geno_exome.columns if c != "IID"]:
    new_participants_exome = df_geno_exome.loc[df_geno_exome[variant] < 2, "IID"].tolist()
    if len(new_participants_exome) > 0:
        if variant not in carrier_lists:
            carrier_lists[variant] = new_participants_exome
        else:
            carrier_lists[variant] = list(set(carrier_lists[variant] + new_participants_exome))


## Pull clinical data

In [ ]:
variant_clinical_data = {}
for variant in carrier_lists:
    df_cov_subset = df_cov[df_cov["IID"].isin(carrier_lists[variant])].copy()
    df_cov_subset = df_cov_subset[df_cov_subset["PHENO"].isin(["PD", "Control"])]
    if len(df_cov_subset) > 0:
        variant_clinical_data[variant] = {
            "AAO_min": df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "age_of_onset"].min(),
            "AAO_median": df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "age_of_onset"].median(),
            "AAO_max": df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "age_of_onset"].max(),
            "AAO_20": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 10) & (df_cov_subset["age_of_onset"] <= 20)]),
            "AAO_30": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 20) & (df_cov_subset["age_of_onset"] <= 30)]),
            "AAO_40": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 30) & (df_cov_subset["age_of_onset"] <= 40)]),
            "AAO_50": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 40) & (df_cov_subset["age_of_onset"] <= 50)]),
            "AAO_60": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 50) & (df_cov_subset["age_of_onset"] <= 60)]),
            "AAO_70": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 60) & (df_cov_subset["age_of_onset"] <= 70)]),
            "AAO_80": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 70) & (df_cov_subset["age_of_onset"] <= 80)]),
            "AAO_90": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 80) & (df_cov_subset["age_of_onset"] <= 90)]),
            "AAO_100": len(df_cov_subset[(df_cov_subset["PHENO"] == "PD") & (df_cov_subset["age_of_onset"] > 90) & (df_cov_subset["age_of_onset"] <= 100)]),
            "Age_min": df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "current_age"].min(),
            "Age_median": df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "current_age"].median(),
            "Age_max": df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "current_age"].max(),
            "Age_10": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] <= 10)]),
            "Age_20": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 10) & (df_cov_subset["current_age"] <= 20)]),
            "Age_30": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 20) & (df_cov_subset["current_age"] <= 30)]),
            "Age_40": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 30) & (df_cov_subset["current_age"] <= 40)]),
            "Age_50": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 40) & (df_cov_subset["current_age"] <= 50)]),
            "Age_60": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 50) & (df_cov_subset["current_age"] <= 60)]),
            "Age_70": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 60) & (df_cov_subset["current_age"] <= 70)]),
            "Age_80": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 70) & (df_cov_subset["current_age"] <= 80)]),
            "Age_90": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 80) & (df_cov_subset["current_age"] <= 90)]),
            "Age_100": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 90) & (df_cov_subset["current_age"] <= 100)]),
            "Age_110": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 100) & (df_cov_subset["current_age"] <= 110)]),
            "Age_120": len(df_cov_subset[(df_cov_subset["PHENO"] == "Control") & (df_cov_subset["current_age"] > 110) & (df_cov_subset["current_age"] <= 120)]),
            "FH_neg_PD": (df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "family_history_for_qc"] == "No").sum(),
            "FH_unk_PD": (df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "family_history_for_qc"] == "Not Reported").sum(),
            "FH_pos_PD": (df_cov_subset.loc[df_cov_subset["PHENO"] == "PD", "family_history_for_qc"] == "Yes").sum(),
            "FH_neg_HC": (df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "family_history_for_qc"] == "No").sum(),
            "FH_unk_HC": (df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "family_history_for_qc"] == "Not Reported").sum(),
            "FH_pos_HC": (df_cov_subset.loc[df_cov_subset["PHENO"] == "Control", "family_history_for_qc"] == "Yes").sum(),
        }
df_variant_clinical_data = pd.DataFrame(variant_clinical_data).T
df_variant_clinical_data = df_variant_clinical_data.reset_index(names=["ID"])
df_variant_clinical_data["POS"] = df_variant_clinical_data["ID"].str.split(":").str[1].astype(int)
df_variant_clinical_data = df_variant_clinical_data.sort_values(by="POS")
df_variant_clinical_data.drop(columns="POS", inplace=True)


## Annotation

### Reformat variants for ANNOVAR

In [ ]:
if not pathlib.Path(f"{WORK_DIR}/annovar_input.tsv").exists():
    # Split ID into separate columns
    df_annovar = df_variant_clinical_data[["ID"]]
    variant_split = df_annovar["ID"].str.split(":", expand=True)
    df_annovar = pd.DataFrame({
        "chr": variant_split[0],
        "start": variant_split[1],
        "end": variant_split[1],
        "ref": variant_split[2],
        "alt": variant_split[3],
    })
    
    # Reformat deletions (add 1 to position, replace alt with "-" and remove first base in ref)
    del_mask = df_annovar["ref"].copy().str.len() > 1
    df_annovar.loc[del_mask, "start"] = df_annovar.loc[del_mask, "start"].astype(int) + 1
    df_annovar.loc[del_mask, "end"] = df_annovar.loc[del_mask, "end"].astype(int) + df_annovar.loc[del_mask, "ref"].str.len() - 1
    df_annovar.loc[del_mask, "ref"] = df_annovar.loc[del_mask, "ref"].str[1:]
    df_annovar.loc[del_mask, "alt"] = "-"
    
    # Reformat insertions (add insertion length to end position, replace ref with "-" and remove first base in alt)
    ins_mask = df_annovar["alt"].copy().str.len() > 1
    df_annovar.loc[ins_mask, "start"] = df_annovar.loc[ins_mask, "start"].astype(int)
    df_annovar.loc[ins_mask, "end"] = df_annovar.loc[ins_mask, "end"].astype(int)
    df_annovar.loc[ins_mask, "ref"] = "-"
    df_annovar.loc[ins_mask, "alt"] = df_annovar.loc[ins_mask, "alt"].str[1:]
    
    # Save annovar input file
    df_annovar.to_csv(f"{WORK_DIR}/annovar_input.tsv", sep="\t", index=False, header=False)


### Run ANNOVAR

In [ ]:
if not pathlib.Path(f"{WORK_DIR}/annotations.hg38_multianno.txt").exists():
    !cd /home/jupyter/annovar && perl ./table_annovar.pl {WORK_DIR}/annovar_input.tsv ./humandb/ \
    -buildver hg38 \
    -out {WORK_DIR}/annotations \
    -remove \
    -protocol refGene,clinvar_20250721,dbnsfp47a,gnomad41_genome,avsnp151 \
    -operation g,f,f,f,f \
    --nopolish \
    -nastring .


### Process annotated data

In [ ]:
# Read annotations and reformat columns
df_anno = pd.read_csv(f"{WORK_DIR}/annotations.hg38_multianno.txt", sep="\t", low_memory=False)
df_anno.insert(0, "ID", df_variant_clinical_data["ID"].tolist())
df_anno[["Chr", "Start", "Ref", "Alt"]] = df_anno["ID"].str.split(":", expand=True)
df_anno["Start"] = df_anno["Start"].astype(int)
df_anno["End"] = df_anno["Start"]


In [ ]:
# Keep only the columns we care about
df_anno = df_anno[[
    "ID",
    "Chr",
    "Start",
    "End",
    "Ref",
    "Alt",
    "Func.refGene",
    "Gene.refGene",
    "GeneDetail.refGene",
    "ExonicFunc.refGene",
    "AAChange.refGene",
    "CADD_phred",
    "CLNSIG",
    "CLNDN",
    "gnomad41_genome_AF",
    "gnomad41_genome_AF_afr",
    "gnomad41_genome_AF_ami",
    "gnomad41_genome_AF_amr",
    "gnomad41_genome_AF_asj",
    "gnomad41_genome_AF_eas",
    "gnomad41_genome_AF_fin",
    "gnomad41_genome_AF_mid",
    "gnomad41_genome_AF_nfe",
    "gnomad41_genome_AF_remaining",
    "gnomad41_genome_AF_sas",
    "avsnp151",
]]


In [ ]:
# Split AA change column by ":" into 5 new columns
df_anno[["gene", "transcript", "exon", "cDNA", "prot_change"]] = (
    df_anno["AAChange.refGene"].str.split(":", expand=True)
)
# df_anno["prot_change"] = df_anno["prot_change"].str.split(".").str[1]


In [ ]:
# Merge conservation and kinase activity data
df_kinase_activity = pd.read_csv(
    f"{WORK_DIR}/kinase_activity.tsv",
    sep="\t",
).rename(columns={"Variant":"prot_change"})
df_kinase_activity["prot_change"] = "p." + df_kinase_activity["prot_change"].astype(str)
df_anno = df_anno.merge(df_kinase_activity, on="prot_change", how="left")


In [ ]:
# Domain start and end positions
lrrk2_domains = {
    "ARM": {"START": 40225618, "END": 40278135},
    "ANK": {"START": 40278136, "END": 40284033},
    "LRR": {"START": 40284034, "END": 40308510},
    "ROC": {"START": 40308511, "END": 40310646},
    "COR": {"START": 40310647, "END": 40323287},
    "KIN": {"START": 40323288, "END": 40351581},
    "WD40": {"START": 40351582, "END": 40367760},
}

# Function to assign domain name based on start position
def assign_domain(pos):
    for domain, coords in lrrk2_domains.items():
        if coords["START"] <= pos <= coords["END"]:
            return domain
    return "Unknown"

# Assign domain names to each variant
df_anno["domain"] = df_anno["Start"].apply(assign_domain)


In [ ]:
# Reformat clinical annotations
df_anno["Clinvar_Pathogenic"] = np.where(df_anno["CLNDN"].str.lower().str.contains("parkinson", na=False), df_anno["CLNSIG"], "")


In [ ]:
df_meta = df_variant_clinical_data.merge(df_anno, on="ID")


# Post-processing

In [ ]:
# Read in combined data table
for modality in modalities:
    df_orig = var_counts_dict[modality].copy()
    ancestry_groups = dict(tuple(df_orig.groupby("Ancestry")))

    # Find all variants present across all ancestries
    df_variants = df_orig[["ID"]].drop_duplicates(subset=["ID"]).copy()
    df_variants["pos"] = df_variants["ID"].str.split(":").str[1].astype(int)
    df_variants.sort_values(by="pos", ascending=True, inplace=True)
    df_variants = df_variants[["ID"]].reset_index(drop=True)
    
    df_final = []
    for anc, df_ancestry in ancestry_groups.items():
        df_all_variants = df_variants.merge(df_ancestry, on="ID", how="left")
        # df_all_variants[["total_PD", "total_HC"]] = df_all_variants[["total_PD", "total_HC"]].apply(
        #     lambda col: col.fillna(col.max())
        # )
        df_all_variants["Ancestry"] = ancestry_mappings[anc]
        df_all_variants = df_all_variants.fillna(0)
        df_final.append(df_all_variants)
    df_final = pd.concat(df_final)

    # Aggregate counts and age min/max/median across ancestries for each variant
    df_combined = df_final.drop(columns="Ancestry").groupby("ID", as_index=False).agg("sum")
    int_cols = [col for col in df_combined.columns.values if col != "ID"]
    df_combined[int_cols] = df_combined[int_cols].astype(int)
    df_combined["Ancestry"] = "Combined"

    # Combine individual ancestry data with combined data
    df_final = [df_combined, df_final]
    df_final = pd.concat(df_final)

    # Merge counts for each ancestry with variant metadata
    df_final = pd.merge(df_final, df_meta, on="ID", how="left")
    df_final.to_csv(f"{WORK_DIR}/results/lrrk2_combined_{modality}.tsv", sep="\t", index=False)


In [ ]:
# Initialize result dict
var_counts = {
    "ID": [],
    "het_PD": [],
    "hom_PD": [],
    "total_PD": [len(df_geno_exome)] * (df_geno_exome.shape[1] - 2),
    "het_HC": [0] * (df_geno_exome.shape[1] - 2),
    "hom_HC": [0] * (df_geno_exome.shape[1] - 2),
    "total_HC": [0] * (df_geno_exome.shape[1] - 2),
}

# Loop through SNPs and calculate counts
for snp in [col for col in df_geno_exome.columns.values if not col in ["IID", "PHENOTYPE"]]:
    var_counts["ID"].append(snp)
    var_counts["het_PD"].append((df_geno_exome[snp] == 1).sum())
    var_counts["hom_PD"].append((df_geno_exome[snp] == 0).sum())

# Convert results to DataFrame
df_var_counts = pd.DataFrame(var_counts)
df_var_counts["Ancestry"] = "Combined"
df_var_counts = pd.merge(df_var_counts, df_meta, on="ID", how="left")
df_var_counts.to_csv(f"{WORK_DIR}/results/lrrk2_combined_exome.tsv", sep="\t", index=False)


In [ ]:
df_wgs = pd.read_csv(f"{WORK_DIR}/results/lrrk2_combined_wgs.tsv", sep="\t")
dfs = []
for i in range(1, 6):
    n_rows = len(df_wgs)
    start = (n_rows * (i-1)) // 5
    end = (n_rows * (i)) // 5
    df_wgs.iloc[start:end, :].to_csv(f"{WORK_DIR}/results/lrrk2_combined_wgs_{i}.tsv", sep="\t", index=False)


# Figures and Tables

## Donut plots

In [ ]:
df_wgs = pd.read_csv(
    f"{WORK_DIR}/results/lrrk2_combined_wgs.tsv",
    sep="\t",
    usecols = ["Ancestry", "het_PD", "hom_PD", "prot_change"],
).rename(columns={"prot_change":"variant"})
df_wgs["n"] = df_wgs["het_PD"] + 2 * df_wgs["hom_PD"]
df_wgs = df_wgs[["Ancestry", "variant", "n"]]

df_colors = pd.DataFrame({
    "variant": ["p.R1067Q", "p.I1122V", "p.R1325Q", "p.S1403R", "p.N1437H", "p.N1437D", "p.N1437S", 
                "p.A1440P", "p.R1441S", "p.R1441G", "p.R1441C", "p.R1441H", "p.A1442P", "p.V1447M", 
                "p.R1628P", "p.I1658F", "p.Y1699C", "p.F1700L", "p.R1728H", "p.S1761R", "p.L1795F", 
                "p.G2019S", "p.I2020L", "p.I2020T", "p.I2020S", "p.T2031S", "p.G2385R"],
    "color": ["#0C8DC3", "#0C8DC3", "#0C8DC3", "#CCCCCC", "#CCCCCC", "#CCCCCC", "#CCCCCC", 
              "#CCCCCC", "#CCCCCC", "#CCCCCC", "#CCCCCC", "#CCCCCC", "#CCCCCC", "#CCCCCC", 
              "#4BA3C9", "#4BA3C9", "#999999", "#999999", "#999999", "#999999", "#999999", 
              "#86C1DA", "#86C1DA", "#86C1DA", "#86C1DA", "#86C1DA", "#666666"],
})

df = df_colors.merge(df_wgs, on="variant")
df = df.groupby(["Ancestry", "variant"]).agg({"n": "sum", "color": "first"}).reset_index()
df = df[df["n"] > 0]

for ancestry in ancestries:
    df_ancestry = df[df["Ancestry"].str.contains(ancestry)]

    if len(df_ancestry) == 0:
        continue
        
    df_ancestry.insert(
        loc=len(df_ancestry.columns), 
        column="pos", 
        value=df_ancestry["variant"].str.extract(r'(\d+)', expand=False)
    )
    df_ancestry.loc[:, "pos"] = pd.to_numeric(df_ancestry.loc[:, "pos"], errors='coerce')
    df_ancestry = df_ancestry.sort_values(by="pos", ascending=True, na_position="last")  
    display(df_ancestry)
    
    OUTFILE = f"{WORK_DIR}/results/{ancestry}_donut.png"
    
    # =========================
    # FIXED GLOBAL SETTINGS
    # =========================
    FIGSIZE = (10, 10)
    STARTANGLE = 90
    CLOCKWISE = True
    
    DONUT_RADIUS = 1.0
    DONUT_WIDTH  = 0.5
    
    LABEL_RADIUS = 1.30  # Fixed radius from center for all labels
    ANCHOR_R = 0.98
    TEXT_PADDING = 0.04
    
    FONT_SIZE = 12
    CENTER_FONT_SIZE = 16
    LINE_COLOR = "#808080"
    LINE_WIDTH = 0.6
    DPI = 600
    
    # Replace MIN_ANGULAR_SPACING with MIN_Y_SPACING
    # 0.12 is usually perfect for fontsize=18 on a 10x10 figure
    MIN_Y_SPACING = 0.12  
    
    # =========================
    # CREATE DONUT
    # =========================
    sizes  = df_ancestry["n"].to_numpy()
    colors = df_ancestry["color"].to_numpy()
    labels = df_ancestry["variant"].to_numpy()
    
    total_carriers = sizes.sum()
    
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.set(aspect="equal")
    
    wedges, _ = ax.pie(
        sizes,
        colors=colors,
        radius=DONUT_RADIUS,
        startangle=STARTANGLE,
        counterclock=not CLOCKWISE,
        wedgeprops=dict(width=DONUT_WIDTH, edgecolor="white"),
    )
    
    # =========================
    # ADD CENTER TEXT
    # =========================
    ax.text(
        0, 0.1, ancestry,
        ha="center", va="center",
        fontsize=CENTER_FONT_SIZE,
        weight="bold",
    )
    ax.text(
        0, -0.1, f"(n = {int(total_carriers)})",
        ha="center", va="center",
        fontsize=CENTER_FONT_SIZE,
    )
    
    # =========================
    # CALCULATE INITIAL POSITIONS
    # =========================
    label_data = []
    for w, lab in zip(wedges, labels):
        # If there is only 1 variant (100% donut), force the label to the 
        # right side (0 degrees) instead of the bottom.
        if len(wedges) == 1:
            ang_deg = 0.0
            ang_rad = 0.0
        else:
            ang_deg = (w.theta1 + w.theta2) / 2
            ang_rad = np.deg2rad(ang_deg)
        
        # Anchor point on donut edge
        anchor_x = ANCHOR_R * np.cos(ang_rad)
        anchor_y = ANCHOR_R * np.sin(ang_rad)
        
        # Normalize angle to [-180, 180] to easily split into left/right hemispheres
        norm_angle = ang_deg % 360
        if norm_angle > 180:
            norm_angle -= 360
            
        is_right = -90 <= norm_angle <= 90
        initial_y = LABEL_RADIUS * np.sin(ang_rad)
        
        label_data.append({
            'label': lab,
            'anchor_x': anchor_x,
            'anchor_y': anchor_y,
            'y': initial_y,
            'is_right': is_right,
        })
    
    # =========================
    # ADJUST Y POSITIONS TO PREVENT OVERLAP
    # =========================
    # Separate hemispheres
    left_labels = [d for d in label_data if not d['is_right']]
    right_labels = [d for d in label_data if d['is_right']]
    
    def adjust_y_spacing(labels_list):
        """Adjust Y coordinates to guarantee vertical text spacing"""
        if not labels_list:
            return
        
        # Sort labels from top to bottom (Y descending)
        labels_list.sort(key=lambda d: d['y'], reverse=True)
        
        # 1. Push labels down if they overlap with the one above them
        for i in range(1, len(labels_list)):
            prev_y = labels_list[i-1]['y']
            curr_y = labels_list[i]['y']
            if prev_y - curr_y < MIN_Y_SPACING:
                labels_list[i]['y'] = prev_y - MIN_Y_SPACING
                
        # 2. If pushing them down pushed the bottom label too far off the circle, shift the whole group up
        if labels_list[-1]['y'] < -LABEL_RADIUS:
            shift_up = -LABEL_RADIUS - labels_list[-1]['y']
            for item in labels_list:
                item['y'] += shift_up
                
        # 3. Calculate new X coordinates based on the adjusted Y coordinates
        for item in labels_list:
            # Cap Y strictly to LABEL_RADIUS to avoid math domain errors
            y_capped = max(min(item['y'], LABEL_RADIUS), -LABEL_RADIUS)
            item['y'] = y_capped
            
            # Place X on the circle rim (using max(0, ...) to prevent floating point NaN errors!)
            x_abs = np.sqrt(max(0, LABEL_RADIUS**2 - y_capped**2))
            item['x'] = x_abs if item['is_right'] else -x_abs

    # Apply adjustments
    adjust_y_spacing(left_labels)
    adjust_y_spacing(right_labels)
    
    all_labels = left_labels + right_labels
    
    # =========================
    # DRAW LABELS AND CONNECTOR LINES
    # =========================
    for item in all_labels:
        # Determine text alignment and add horizontal padding outward
        if item['is_right']:
            ha = "left"
            text_x = item['x'] + TEXT_PADDING
        else:
            ha = "right"
            text_x = item['x'] - TEXT_PADDING
        
        # Draw text at the padded X coordinate
        ax.text(
            text_x, item['y'], item['label'],
            ha=ha, va="center",
            fontsize=FONT_SIZE,
        )
        
        # Draw connector line ending exactly at the unpadded 'x' coordinate
        ax.plot(
            [item['anchor_x'], item['x']],
            [item['anchor_y'], item['y']],
            lw=LINE_WIDTH,
            color=LINE_COLOR,
            zorder=0,
        )
    
    # =========================
    # FORCE CONSISTENT IMAGE SIZING
    # =========================
    # 1. Set fixed axis limits so the donut is exactly the same size in every plot
    MAX_EXTENT = 2.5  # Increase this if long text gets cut off on the sides
    ax.set_xlim(-MAX_EXTENT, MAX_EXTENT)
    ax.set_ylim(-MAX_EXTENT, MAX_EXTENT)
    
    # 2. Draw invisible dots at the extreme corners. 
    # This tricks bbox_inches="tight" into cropping every image to the exact 
    # same MAX_EXTENT square, preventing aspect ratio shifting.
    ax.scatter([-MAX_EXTENT, MAX_EXTENT], [-MAX_EXTENT, MAX_EXTENT], alpha=0.0)

    ax.set_axis_off()
    
    # Remove tight_layout() as it interferes with fixed bounding boxes
    # plt.tight_layout() 
    
    plt.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", transparent=True)
    plt.close(fig)


## Sample counts stratified by ancestry, modality, and phenotype

In [ ]:
ancestries = [
    "AAC", 
    "AFR", 
    "AJ", 
    "AMR", 
    "CAH", 
    "CAS", 
    "EAS", 
    "FIN", 
    "MDE", 
    "SAS",
    "EUR",
]
modalities = [
    "wgs",
    "raw",
    "imputed",
]


In [ ]:
rows = []
for ancestry in ancestries:
    for modality in modalities:
        df = pd.read_csv(pathlib.Path(
            pathlib.Path.home(),
            "workspace",
            "ws_files",
            "2026_02_05_smg",
            "results",
            ancestry,
            f"LRRK2_{modality}_{ancestry}.psam"
        ), sep="\t")
        rows.append({
            "ancestry": ancestry,
            "group":    "Cases",
            "modality": modality,
            "count":    (df["PHENO"] == 2).sum(),
        })
        rows.append({
            "ancestry": ancestry,
            "group":    "Controls",
            "modality": modality,
            "count":    (df["PHENO"] == 1).sum(),
        })

result = (
    pd.DataFrame(rows)
    .pivot(index=["ancestry", "group"], columns="modality", values="count")
    .rename_axis(columns=None)
).rename(columns={"imputed": "Imputed genotyping", "raw": "Raw genotyping", "wgs": "WGS"})

combined = (
    result
    .groupby(level="group")
    .sum()
    .assign(ancestry="Combined")
    .set_index("ancestry", append=True)
    .swaplevel()
)
result = pd.concat([result, combined])

df_exomes = pd.read_csv(pathlib.Path(
    pathlib.Path.home(),
    "workspace",
    "ws_files",
    "2026_02_05_smg",
    "results",
    "exomes",
    f"LRRK2.psam",
), sep="\t")


exome_counts = {
    ("Combined", "Cases"):    (df_exomes["PHENO"] == 2).sum(),
    ("Combined", "Controls"): (df_exomes["PHENO"] == 1).sum(),
}
result["Exomes"] = pd.Series(exome_counts)
result.to_csv(pathlib.Path(
    pathlib.Path.home(),
    "workspace",
    "ws_files",
    "2026_02_05_smg",
    "results",
    "table1.tsv"
), sep="\t")
